# Entry-Level Introduction to Qlib

This notebook walks through one complete Qlib workflow for a new user: initialize data, inspect the data layer, build an `Alpha158` dataset, train a LightGBM model, generate predictions, run a backtest, and inspect the results.

**Prerequisites**

- Install `pyqlib[analysis]` in the environment that will run this notebook.
- Make CN daily data available either in a repo-local `qlib_data/cn_data` directory or under `~/.qlib/qlib_data/cn_data`.
- Persisted notebook artifacts in this workflow are intentionally saved with a `demo_` prefix so they are easy to distinguish from other runs.


In [1]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import qlib
from IPython.display import display

from qlib.backtest import backtest as normal_backtest
from qlib.constant import REG_CN
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.evaluate import risk_analysis
from qlib.contrib.eva.alpha import calc_ic
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.tests.data import GetData
from qlib.utils import (
    class_casting,
    exists_qlib_data,
    fill_placeholder,
    flatten_dict,
    get_date_by_shift,
    init_instance_by_config,
)
from qlib.utils.data import deepcopy_basic_type
from qlib.workflow import R
from qlib.workflow.record_temp import PortAnaRecord, SigAnaRecord, SignalRecord

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
pd.options.display.max_rows = 12
pd.options.display.max_columns = 12


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
MARKET = "csi300"
BENCHMARK = "SH000300"
EXPERIMENT_NAME = "entry_level_introduction"

DATA_START, DATA_END = "2008-01-01", "2020-08-01"
TRAIN_START, TRAIN_END = "2008-01-01", "2014-12-31"
VALID_START, VALID_END = "2015-01-01", "2016-12-31"
TEST_START, TEST_END = "2017-01-01", "2020-08-01"

DEMO_MODEL_ARTIFACT = "demo_trained_model.pkl"
DEMO_PRED_ARTIFACT = "demo_pred.pkl"
DEMO_LABEL_ARTIFACT = "demo_label.pkl"
DEMO_IC_ARTIFACT = "demo_ic.pkl"
DEMO_RIC_ARTIFACT = "demo_ric.pkl"
DEMO_REPORT_ARTIFACT = "demo_report_normal_1day.pkl"
DEMO_POSITIONS_ARTIFACT = "demo_positions_normal_1day.pkl"
DEMO_PORT_ANALYSIS_ARTIFACT = "demo_port_analysis_1day.pkl"


def find_repo_root() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (base / "examples").exists() and (base / "qlib").exists():
            return base
    raise FileNotFoundError("Could not locate the qlib repo root from the current working directory.")


REPO_ROOT = find_repo_root()


def resolve_provider_uri() -> Path:
    candidates = []
    env_provider = os.getenv("QLIB_PROVIDER_URI")
    if env_provider:
        candidates.append(Path(env_provider).expanduser().resolve())

    for base in [REPO_ROOT, Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidates.append((base / "qlib_data" / "cn_data").resolve())

    candidates.append(Path("~/.qlib/qlib_data/cn_data").expanduser().resolve())

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        candidate_str = str(candidate)
        if candidate_str not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate_str)

    for candidate in unique_candidates:
        if exists_qlib_data(str(candidate)):
            return candidate

    target = unique_candidates[-1]
    target.parent.mkdir(parents=True, exist_ok=True)
    GetData().qlib_data(target_dir=str(target), region=REG_CN, exists_skip=True)
    return target


provider_uri = resolve_provider_uri()
qlib.init(provider_uri=str(provider_uri), region=REG_CN)
print(f"Qlib {qlib.__version__} initialized with data at: {provider_uri}")


## 1. Inspect the data layer

Before training a model, it helps to see the building blocks that Qlib exposes: the trading calendar, the dynamic stock universe, raw price fields, and expression-based features.


In [ ]:
calendar = pd.Index(D.calendar(start_time="2020-01-01", end_time="2020-03-31"), name="datetime")
csi300_config = D.instruments(MARKET)
csi300_spans = D.list_instruments(csi300_config, start_time="2020-01-01", end_time="2020-03-31")

print("First five trading days:", list(calendar[:5]))
print("Sample constituent spans:")
list(csi300_spans.items())[:5]


In [ ]:
active_counts = pd.Series(
    {
        dt: sum(any(start <= dt <= end for start, end in spans) for spans in csi300_spans.values())
        for dt in calendar
    }
)

ax = active_counts.plot(figsize=(10, 4), title="Active CSI300 constituents by trading day")
ax.set_ylabel("Stocks")
ax.set_xlabel("")
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.show()

active_counts.describe().round(1)


In [ ]:
moutai = D.features(
    ["SH600519"],
    ["$open", "$high", "$low", "$close", "$volume", "$factor"],
    start_time="2020-01-01",
    end_time="2020-02-15",
)
moutai.head()


In [ ]:
moutai_plot = moutai.droplevel("instrument").copy()
for field in ["open", "high", "low", "close"]:
    moutai_plot[f"raw_{field}"] = moutai_plot[f"${field}"] / moutai_plot["$factor"]

fig = go.Figure(
    data=[
        go.Candlestick(
            x=moutai_plot.index,
            open=moutai_plot["raw_open"],
            high=moutai_plot["raw_high"],
            low=moutai_plot["raw_low"],
            close=moutai_plot["raw_close"],
            name="raw price",
        )
    ]
)
fig.update_layout(
    title="SH600519 raw OHLC reconstructed with $factor",
    xaxis_title="Date",
    yaxis_title="Price",
)
fig.show()

ax = moutai_plot[["$close", "raw_close"]].rename(
    columns={"$close": "adjusted_close", "raw_close": "raw_close"}
).plot(figsize=(10, 4), title="Adjusted vs raw close")
ax.set_ylabel("Price")
ax.set_xlabel("")
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.show()


Qlib's expression engine lets you define factors as formulas over raw fields. That becomes useful both for quick exploration and for building larger feature sets such as `Alpha158`.


In [ ]:
ema_spread = D.features(
    ["SH600519"],
    ["(EMA($close, 12) - EMA($close, 26)) / $close"],
    start_time="2020-01-01",
    end_time="2020-06-30",
)
ema_spread.columns = ["ema_spread"]
ema_spread.head()


## 2. Build an `Alpha158` handler

A `DataHandler` bundles feature generation, preprocessing, and label construction into one reusable object. For this introduction notebook we use the built-in `Alpha158` feature set and the same date ranges used by the LightGBM benchmark config.


In [ ]:
handler = Alpha158(
    instruments=MARKET,
    start_time=DATA_START,
    end_time=DATA_END,
    fit_start_time=TRAIN_START,
    fit_end_time=TRAIN_END,
)

feature_slice = handler.fetch(selector=slice("2019-01-01", "2019-01-10"), level="datetime", col_set="feature")
label_slice = handler.fetch(selector=slice("2019-01-01", "2019-01-10"), level="datetime", col_set="label")

print(f"Feature slice shape: {feature_slice.shape}")
print(f"Label slice shape: {label_slice.shape}")
feature_slice.head()


In [ ]:
feature_formulas, feature_names = handler.get_feature_config()

display(pd.DataFrame({"feature": feature_names[:10], "formula": feature_formulas[:10]}))
display(
    pd.Series(
        {
            "feature_missing_ratio": round(feature_slice.isna().mean().mean(), 4),
            "label_missing_ratio": round(label_slice.isna().mean().mean(), 4),
        }
    ).to_frame("value")
)
display(label_slice.head())


`fit_start_time` and `fit_end_time` define the period used to fit processors such as normalization. The handler then reuses those fitted transformations when it serves train, validation, and test data.


In [ ]:
dataset = DatasetH(
    handler=handler,
    segments={
        "train": (TRAIN_START, TRAIN_END),
        "valid": (VALID_START, VALID_END),
        "test": (TEST_START, TEST_END),
    },
)

segment_rows = []
for segment in ["train", "valid", "test"]:
    frame = dataset.prepare(segment)
    segment_rows.append({"segment": segment, "rows": frame.shape[0], "columns": frame.shape[1]})

segment_summary = pd.DataFrame(segment_rows).set_index("segment")
display(segment_summary)

timeline = pd.DataFrame(
    {
        "segment": ["train", "valid", "test"],
        "start": pd.to_datetime([TRAIN_START, VALID_START, TEST_START]),
        "end": pd.to_datetime([TRAIN_END, VALID_END, TEST_END]),
    }
)

fig, ax = plt.subplots(figsize=(10, 2.5))
for _, row in timeline.iterrows():
    ax.hlines(y=row["segment"], xmin=row["start"], xmax=row["end"], linewidth=10)

ax.set_title("Time-based splits used by DatasetH")
ax.set_ylabel("")
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.show()


## 3. Train the model and save demo-prefixed artifacts

The next cell defines lightweight notebook-local record classes that keep Qlib's behavior but save demo-prefixed artifacts. That makes it clear which files came from this introductory notebook.


In [ ]:
class DemoSignalRecord(SignalRecord):
    pred_name = DEMO_PRED_ARTIFACT
    label_name = DEMO_LABEL_ARTIFACT

    def generate(self, **kwargs):
        pred = self.model.predict(self.dataset)
        if isinstance(pred, pd.Series):
            pred = pred.to_frame("score")
        self.save(**{self.pred_name: pred})

        raw_label = self.generate_label(self.dataset)
        if raw_label is not None:
            self.save(**{self.label_name: raw_label})

        return pred

    def list(self):
        return [self.pred_name, self.label_name]


class DemoSigAnaRecord(SigAnaRecord):
    depend_cls = DemoSignalRecord

    def _generate(self, label: pd.DataFrame | None = None, **kwargs):
        pred = self.load(DEMO_PRED_ARTIFACT)
        if label is None:
            label = self.load(DEMO_LABEL_ARTIFACT)
        if label is None or label.empty:
            raise ValueError("DemoSigAnaRecord requires a non-empty label dataframe.")

        ic, ric = calc_ic(pred.iloc[:, 0], label.iloc[:, self.label_col])
        metrics = {
            "IC": ic.mean(),
            "ICIR": ic.mean() / ic.std(),
            "Rank IC": ric.mean(),
            "Rank ICIR": ric.mean() / ric.std(),
        }
        self.recorder.log_metrics(**metrics)
        return {DEMO_IC_ARTIFACT: ic, DEMO_RIC_ARTIFACT: ric}

    def list(self):
        return [DEMO_IC_ARTIFACT, DEMO_RIC_ARTIFACT]


class DemoPortAnaRecord(PortAnaRecord):
    depend_cls = DemoSignalRecord

    def _generate(self, **kwargs):
        pred = self.load(DEMO_PRED_ARTIFACT)

        placeholder_value = {"<PRED>": pred}
        for attr_name in ("executor_config", "strategy_config"):
            setattr(self, attr_name, fill_placeholder(getattr(self, attr_name), placeholder_value))

        dt_values = pred.index.get_level_values("datetime")
        if self.backtest_config["start_time"] is None:
            self.backtest_config["start_time"] = dt_values.min()
        if self.backtest_config["end_time"] is None:
            self.backtest_config["end_time"] = get_date_by_shift(dt_values.max(), -1)

        artifact_objects = {}
        portfolio_metric_dict, _ = normal_backtest(
            executor=self.executor_config,
            strategy=self.strategy_config,
            **self.backtest_config,
        )

        for freq, (report_normal, positions_normal) in portfolio_metric_dict.items():
            artifact_objects[f"demo_report_normal_{freq}.pkl"] = report_normal
            artifact_objects[f"demo_positions_normal_{freq}.pkl"] = positions_normal

        for analysis_freq in self.risk_analysis_freq:
            report_normal, _ = portfolio_metric_dict[analysis_freq]
            analysis = {
                "excess_return_without_cost": risk_analysis(
                    report_normal["return"] - report_normal["bench"],
                    freq=analysis_freq,
                ),
                "excess_return_with_cost": risk_analysis(
                    report_normal["return"] - report_normal["bench"] - report_normal["cost"],
                    freq=analysis_freq,
                ),
            }
            analysis_df = pd.concat(analysis)
            analysis_dict = flatten_dict(analysis_df["risk"].unstack().T.to_dict())
            self.recorder.log_metrics(**{f"{analysis_freq}.{k}": v for k, v in analysis_dict.items()})
            artifact_objects[f"demo_port_analysis_{analysis_freq}.pkl"] = analysis_df

        return artifact_objects

    def list(self):
        paths = []
        for freq in self.all_freq:
            paths.extend(
                [
                    f"demo_report_normal_{freq}.pkl",
                    f"demo_positions_normal_{freq}.pkl",
                ]
            )
        for analysis_freq in self.risk_analysis_freq:
            if analysis_freq in self.all_freq:
                paths.append(f"demo_port_analysis_{analysis_freq}.pkl")
        return paths


task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.2,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": {
                    "instruments": MARKET,
                    "start_time": DATA_START,
                    "end_time": DATA_END,
                    "fit_start_time": TRAIN_START,
                    "fit_end_time": TRAIN_END,
                },
            },
            "segments": {
                "train": (TRAIN_START, TRAIN_END),
                "valid": (VALID_START, VALID_END),
                "test": (TEST_START, TEST_END),
            },
        },
    },
}

model = init_instance_by_config(task["model"])
port_analysis_config = {
    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True,
        },
    },
    "strategy": {
        "class": "TopkDropoutStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "signal": "<PRED>",
            "topk": 50,
            "n_drop": 5,
        },
    },
    "backtest": {
        "start_time": TEST_START,
        "end_time": TEST_END,
        "account": 100000000,
        "benchmark": BENCHMARK,
        "exchange_kwargs": {
            "freq": "day",
            "limit_threshold": 0.095,
            "deal_price": "close",
            "open_cost": 0.0005,
            "close_cost": 0.0015,
            "min_cost": 5,
        },
    },
}


In [ ]:
with R.start(experiment_name=EXPERIMENT_NAME):
    R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(**{DEMO_MODEL_ARTIFACT: model})

    recorder = R.get_recorder()
    DemoSignalRecord(model=model, dataset=dataset, recorder=recorder).generate()
    DemoSigAnaRecord(recorder, ana_long_short=False).generate()
    DemoPortAnaRecord(
        recorder,
        config=port_analysis_config,
        risk_analysis_freq="day",
        indicator_analysis_freq=[],
    ).generate()

    demo_recorder_id = recorder.id

print(f"Recorder ID: {demo_recorder_id}")


## 4. Load the demo-prefixed results

The next cells load only the demo-prefixed artifacts created above. That keeps the notebook self-contained and makes the persisted outputs easy to identify in the recorder.


In [ ]:
recorder = R.get_recorder(experiment_name=EXPERIMENT_NAME, recorder_id=demo_recorder_id)

pred_df = recorder.load_object(DEMO_PRED_ARTIFACT)
label_df = recorder.load_object(DEMO_LABEL_ARTIFACT)
ic = recorder.load_object(f"sig_analysis/{DEMO_IC_ARTIFACT}")
ric = recorder.load_object(f"sig_analysis/{DEMO_RIC_ARTIFACT}")
report_normal_df = recorder.load_object(f"portfolio_analysis/{DEMO_REPORT_ARTIFACT}")
positions = recorder.load_object(f"portfolio_analysis/{DEMO_POSITIONS_ARTIFACT}")
analysis_df = recorder.load_object(f"portfolio_analysis/{DEMO_PORT_ANALYSIS_ARTIFACT}")

artifact_listing = pd.concat(
    [
        pd.Series(sorted(recorder.list_artifacts()), name="root"),
        pd.Series(sorted(recorder.list_artifacts("sig_analysis")), name="sig_analysis"),
        pd.Series(sorted(recorder.list_artifacts("portfolio_analysis")), name="portfolio_analysis"),
    ],
    axis=1,
)
artifact_listing


In [ ]:
display(pred_df.head())

sample_date = pred_df.index.get_level_values("datetime").unique()[20]
daily_scores = pred_df.xs(sample_date, level="datetime").sort_values("score", ascending=False)
top_bottom = pd.concat(
    [daily_scores.head(5).assign(bucket="top_5"), daily_scores.tail(5).assign(bucket="bottom_5")]
).reset_index()
display(top_bottom)

metric_summary = pd.Series(
    {
        "IC": ic.mean(),
        "ICIR": ic.mean() / ic.std(),
        "Rank IC": ric.mean(),
        "Rank ICIR": ric.mean() / ric.std(),
        "Annualized excess return (with cost)": analysis_df.loc[
            ("excess_return_with_cost", "annualized_return"),
            "risk",
        ],
        "Information ratio (with cost)": analysis_df.loc[
            ("excess_return_with_cost", "information_ratio"),
            "risk",
        ],
        "Max drawdown (with cost)": analysis_df.loc[
            ("excess_return_with_cost", "max_drawdown"),
            "risk",
        ],
    }
).to_frame("value")
metric_summary.style.format("{:.4f}")


In [ ]:
pred_label = pd.concat(
    [
        pred_df,
        label_df.iloc[:, [0]].rename(columns={label_df.columns[0]: "label"}),
    ],
    axis=1,
    join="inner",
).dropna()
pred_label.head()


In [ ]:
analysis_position.score_ic_graph(pred_label)
analysis_model.model_performance_graph(pred_label, graph_names=["group_return"])


In [ ]:
analysis_position.report_graph(report_normal_df)
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)


## 5. Move from the notebook to `qrun`

Once the notebook story makes sense, the standard LightGBM benchmark config is the easiest way to rerun the same workflow from the command line.


In [ ]:
workflow_config_path = REPO_ROOT / "examples" / "benchmarks" / "LightGBM" / "workflow_config_lightgbm_Alpha158.yaml"
print(f"Reference config: {workflow_config_path}")
print()
print("Update this field in the YAML before running it from the terminal:")
print(f"  qlib_init.provider_uri: {provider_uri}")
print()
print(f"Command: qrun {workflow_config_path}")
